# Workbook — minimal input, unrolled pipeline

User-facing input is tiny: just `n_stations`, `n_trips`, `n_days`.
The mock emits only `df_stations` + `df_trips`; every other optional
source field is `None`.  `DataLoaderGraph` synthesizes the skeleton
that `build_model` needs, and `build_model` derives the rest
(`periods`, `facility_roles`, `demand`, `supply`, and seeds
`inventory_initial` from first-day outflow).

1. Imports
2. Minimal user config
3. Mock source — `mock.load_data()` (stations + trips only)
4. Source → RawModelData (unrolled `DataLoaderGraph._build_raw_model`)
5. RawModelData → ResolvedModelData (unrolled `build_model`)
6. Simulation
7. Interactive dashboard

In [ ]:
# %% 1. Imports
import sys                                               # path bootstrap
import dataclasses                                       # dataclasses.replace for raw updates
from pathlib import Path                                 # repo root discovery

for _candidate in [Path.cwd()] + list(Path.cwd().parents):
    if (_candidate / "gbp").is_dir() and (_candidate / "tests").is_dir():
        sys.path.insert(0, str(_candidate))
        break

import numpy as np                                       # numerics for the dashboard
import pandas as pd                                      # all tables are DataFrames
import plotly.graph_objects as go                        # map rendering
import gradio as gr                                      # interactive dashboard

from gbp.core.attributes.registry import AttributeRegistry
from gbp.core.model import RawModelData, ResolvedModelData
from gbp.loaders import DataLoaderGraph, DataLoaderMockMinimal, GraphLoaderConfig
from gbp.loaders.contracts import StationsSourceSchema  # only stations are present

from gbp.build.defaults import (                         # derivation helpers
    default_commodity_categories, default_resource_categories,
    derive_demand_from_flow, derive_facility_roles,
    derive_inventory_from_flow, derive_inventory_initial,
    derive_supply_from_flow,
)
from gbp.build.edge_builder import build_edges
from gbp.build.fleet_capacity import compute_fleet_capacity
from gbp.build.lead_time import resolve_lead_times
from gbp.build.pipeline import _ensure_edges_and_commodities
from gbp.build.report import BuildReport
from gbp.build.spine import assemble_spines
from gbp.build.time_resolution import (
    build_periods_from_segments, resolve_all_time_varying, resolve_registry_attributes,
)
from gbp.build.transformation import resolve_transformations
from gbp.build.validation import validate_raw_model

from gbp.consumers.simulator import (
    Environment, EnvironmentConfig, OrganicFlowPhase,
)


RIDEABLE_TYPES: tuple[str, ...] = ("electric_bike", "classic_bike")
MEMBER_CASUAL: tuple[str, ...] = ("member", "casual")
STATION_NAME_POOL: tuple[str, ...] = (
    "Bond St & Bergen St",
    "Amity St & Court St",
    "Norman Ave & Leonard St",
    "Jackson Ave & 49 Ave",
    "E 2 St & Ave C",
    "E 2 St & 2 Ave",
    "St James Pl & Oliver St",
    "W 4 St & 7 Ave S",
    "Broadway & W 51 St",
    "Grand Army Plaza & Central Park S",
    "Front St & Gold St",
    "Henry St & Grand St",
)

TRIP_COLUMNS: tuple[str, ...] = (
    "ride_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "start_station_name",
    "start_station_id",
    "end_station_name",
    "end_station_id",
    "start_lat",
    "start_lng",
    "end_lat",
    "end_lng",
    "member_casual",
)
STATION_COLUMNS: tuple[str, ...] = ("station_id", "station_name", "lat", "lon")

# %% 2. Minimal user config
# The whole user-facing surface: how many stations, how many trips, how many
# days of history.  The mock emits only df_stations + df_trips — everything
# else (periods, demand, supply, inventory seed) is derived downstream.
mock_config = {
    "n_stations": 8,                                     # bike stations in the mock network
    "n_trips": 200,                                      # total trips over the horizon
    "n_days": 3,                                         # span of the trip history in days
}

# %% 3. Mock source — stations + trips only
# DataLoaderMockMinimal implements BikeShareSourceProtocol but only populates
# df_stations (station_id + lat/lon) and df_trips (start/end/started_at).
# Every other optional field is None; the loader and build pipeline detect
# absence and fall back to derivations.
mock = DataLoaderMockMinimal(mock_config)
n_stations = int(mock.config.get("n_stations", 8))
n_trips = int(mock.config.get("n_trips", 200))
n_days = int(mock.config.get("n_days", 3))
start_date = pd.Timestamp(mock.config.get("start_date", "2025-01-01"))
lat_lo, lat_hi = mock.config.get("lat_range", (40.70, 40.80))
lon_lo, lon_hi = mock.config.get("lon_range", (-74.02, -73.92))
min_trip = int(mock.config.get("min_trip_minutes", 3))
max_trip = int(mock.config.get("max_trip_minutes", 45))

# Stations — real Citi Bike ids look like "4404.10"; we mimic that shape.
station_ids = np.array(
    [f"{4000 + i * 13}.{(i % 20) + 1:02d}" for i in range(n_stations)]
)
station_names = np.array(
    [STATION_NAME_POOL[i % len(STATION_NAME_POOL)] for i in range(n_stations)]
)
station_lats = mock._rng.uniform(lat_lo, lat_hi, size=n_stations)
station_lons = mock._rng.uniform(lon_lo, lon_hi, size=n_stations)

mock.df_stations = pd.DataFrame({
    "station_id": station_ids,
    "station_name": station_names,
    "lat": station_lats,
    "lon": station_lons,
})

# Trip endpoints (ensure start != end).
start_idx = mock._rng.integers(0, n_stations, size=n_trips)
end_idx = (start_idx + mock._rng.integers(1, n_stations, size=n_trips)) % n_stations

# Timestamps vectorized via pd.to_timedelta.
offset_minutes = mock._rng.integers(0, n_days * 24 * 60, size=n_trips)
duration_minutes = mock._rng.integers(min_trip, max_trip + 1, size=n_trips)
started_at = start_date + pd.to_timedelta(offset_minutes, unit="m")
ended_at = started_at + pd.to_timedelta(duration_minutes, unit="m")

# 16-char uppercase hex ride_ids, matching the CSV shape.
hi = mock._rng.integers(0, 2**32, size=n_trips, dtype=np.uint32)
lo = mock._rng.integers(0, 2**32, size=n_trips, dtype=np.uint32)
ride_ids = [f"{int(h):08X}{int(l):08X}" for h, l in zip(hi, lo)]

rideable_types = mock._rng.choice(RIDEABLE_TYPES, size=n_trips)
member_casual = mock._rng.choice(MEMBER_CASUAL, size=n_trips)

mock.df_trips = pd.DataFrame({
    "ride_id": ride_ids,
    "rideable_type": rideable_types,
    "started_at": started_at,
    "ended_at": ended_at,
    "start_station_name": station_names[start_idx],
    "start_station_id": station_ids[start_idx],
    "end_station_name": station_names[end_idx],
    "end_station_id": station_ids[end_idx],
    "start_lat": station_lats[start_idx],
    "start_lng": station_lons[start_idx],
    "end_lat": station_lats[end_idx],
    "end_lng": station_lons[end_idx],
    "member_casual": member_casual,
}).sort_values("started_at").reset_index(drop=True)

# %% 4. Source → RawModelData — unrolled loader
# Equivalent to `raw = DataLoaderGraph(mock).load()`.  Split out so every step
# is visible.  The resulting RawModelData is intentionally minimal: periods,
# facility_roles, demand, supply, inventory_initial, resource_categories are
# NOT populated here — build_model derives them in section 5.
loader = DataLoaderGraph(mock, GraphLoaderConfig())

# 4.1 Source schema validation — only stations; depots/resources are absent.
StationsSourceSchema.validate(loader._source.df_stations)

# 4.2 Temporal — planning_horizon inferred from df_trips date range.
# Here i am building 2 dataframes. In the first one start and end dates. 
# But in the second one start and end dates and segment index since sometimes we need to split the horizont on different segments (day, hour)
# В существующей модели лучше оставить единую дневную гранулярность, так как спрос ночью втечении пары часов мал - это маленькая погрешность.
# И при этом ребалансировщик это ТАСК! А у таска своё время и может быть часом или ещё гранулярнее. 
# Тоесть симулятор живёт в своём времени, а ТАСК в своём
temporal = loader._build_temporal()

# 4.3 Entities — facilities only (no depots), commodity_categories from trips.
# Зачем нужны unit в commodity_categories?
entities = loader._build_entities()

# 4.4 Behavior — facility_operations and station↔station edge_rule only.
# В facility_operations нет гранулярности commodities. 
# Хотя чисто теоретически фасилити может принимать велики, но не держать велики. Или ещё что то..
behavior = loader._build_behavior(entities)

# 4.5 Edges — all-pairs distance matrix over stations.
distance_data = (
    loader._build_distance_matrix(entities) if loader._config.build_edges else {}
)

# 4.6 Resources — skipped entirely (no depots + no df_resources → empty dict).
resources = loader._build_resources(entities)

# 4.7 Node parameters — no inventory_initial, no capacity registration.
registry = AttributeRegistry()
flow_data = loader._build_node_parameters(entities, registry)

# 4.8 Costs — no station/depot/truck cost tables, so the registry stays empty.
loader._register_facility_costs(registry, temporal)
loader._register_resource_costs(registry, entities)

# 4.9 Observations — trips → observed_flow (no telemetry → no observed_inventory).
observations = loader._build_observations(entities)

# 4.10 Assemble minimal RawModelData.
raw_tables = {
    **temporal,
    **entities.tables,
    **behavior,
    **distance_data,
    **flow_data,
    **resources,
    **{k: v for k, v in observations.items() if v is not None},
}
raw = RawModelData(
    **{k: v for k, v in raw_tables.items() if v is not None},
    attributes=registry,
)

# %% 5. RawModelData → ResolvedModelData — unrolled build_model
# Equivalent to `resolved = build_model(raw)`.  Every step kept as a separate
# intermediate variable so the data flow through the build pipeline is visible.

# 5.1 Derivations — fill in what the user / loader did not provide.
#     Mirrors gbp.build.pipeline._apply_derivations and records every
#     filled-in table in a BuildReport.
build_report = BuildReport()
derivation_updates: dict[str, pd.DataFrame] = {}

if raw.periods is None:
    derivation_updates["periods"] = build_periods_from_segments(
        raw.planning_horizon, raw.planning_horizon_segments,
    )
    build_report.add("periods", "built from planning_horizon_segments")

if raw.commodity_categories is None:
    derivation_updates["commodity_categories"] = default_commodity_categories()
    build_report.add("commodity_categories", "synthesized default single category")

if raw.resource_categories is None:
    has_resource_data = (
        (raw.resource_fleet is not None and not raw.resource_fleet.empty)
        or (raw.resources is not None and not raw.resources.empty)
    )
    if has_resource_data:
        derivation_updates["resource_categories"] = default_resource_categories()
        build_report.add("resource_categories", "synthesized default single category")

if raw.facility_roles is None:
    derivation_updates["facility_roles"] = derive_facility_roles(
        raw.facilities, raw.facility_operations,
    )
    build_report.add(
        "facility_roles",
        "derived from facility_type + facility_operations via derive_roles()",
    )

if raw.demand is None and raw.observed_flow is not None and not raw.observed_flow.empty:
    derivation_updates["demand"] = derive_demand_from_flow(raw.observed_flow)
    build_report.add("demand", "derived from observed_flow (groupby source_id x date x cc)")

if raw.supply is None and raw.observed_flow is not None and not raw.observed_flow.empty:
    derivation_updates["supply"] = derive_supply_from_flow(raw.observed_flow)
    build_report.add("supply", "derived from observed_flow (groupby target_id x date x cc)")

if raw.inventory_initial is None:
    if raw.observed_inventory is not None and not raw.observed_inventory.empty:
        derivation_updates["inventory_initial"] = derive_inventory_initial(raw.observed_inventory)
        build_report.add(
            "inventory_initial",
            "derived from first observed snapshot per facility x commodity_category",
        )
    elif raw.observed_flow is not None and not raw.observed_flow.empty:
        seeded = derive_inventory_from_flow(raw.observed_flow)
        if not seeded.empty:
            derivation_updates["inventory_initial"] = seeded
            build_report.add(
                "inventory_initial",
                "seeded from first-day outflow in observed_flow "
                "(no observed_inventory telemetry)",
            )

raw_derived = dataclasses.replace(raw, **derivation_updates) if derivation_updates else raw

# 5.2 Validation — schema shape + referential integrity + graph connectivity.
raw_derived.validate()
validate_raw_model(raw_derived).raise_if_invalid()

# 5.3 Time resolution — date → period_id on every time-varying table.
periods = raw_derived.periods.copy()
resolved_time = resolve_all_time_varying(raw_derived, periods)

# 5.4 Registry attribute resolution — date → period_id inside the AttributeRegistry.
resolved_attrs = resolve_registry_attributes(raw_derived.attributes, periods)

# 5.5 Edges — use raw.edges if set, otherwise build from rules + distance_matrix.
edges_df, edge_commodities_df = _ensure_edges_and_commodities(raw_derived)

# 5.6 Edge lead times — hours → periods per edge x period.
edge_lead_time_resolved: pd.DataFrame | None = None
if edges_df is not None and not edges_df.empty:
    edge_lead_time_resolved = resolve_lead_times(edges_df, periods)
    if edge_lead_time_resolved.empty:
        edge_lead_time_resolved = None

# 5.7 Transformations — N:M commodity conversion (empty here — no transforms).
transformation_resolved = resolve_transformations(
    raw_derived.facilities,
    raw_derived.transformations,
    raw_derived.transformation_inputs,
    raw_derived.transformation_outputs,
)

# 5.8 Fleet capacity — None in the minimal case (no fleet / no resource_categories).
fleet_capacity = compute_fleet_capacity(
    raw_derived.resource_fleet,
    raw_derived.resource_categories,
    raw_derived.resources,
)

# 5.9 Assemble ResolvedModelData from raw + build artifacts.
resolved = ResolvedModelData.from_raw(
    raw_derived,
    periods=periods,
    resolved_time=resolved_time,
    resolved_attrs=resolved_attrs,
    edges=edges_df,
    edge_commodities=edge_commodities_df,
    edge_lead_time_resolved=edge_lead_time_resolved,
    transformation_resolved=transformation_resolved,
    fleet_capacity=fleet_capacity,
)

# 5.10 Spines — grain-grouped attribute DataFrames for vectorized lookups.
spines = assemble_spines(resolved)
resolved.facility_spines = spines["facility"] or None
resolved.edge_spines = spines["edge"] or None
resolved.resource_spines = spines["resource"] or None
resolved.build_report = build_report

# %% 6. Simulation
# Environment replays observed_flow row-by-row via OrganicFlowPhase, so
# simulation_flow_log ends up equal to resolved.observed_flow (plus the
# period_index / phase_name columns added by the log writer).  Inventory is
# updated on both sides (− at source, + at target) and clipped at zero.
env_config = EnvironmentConfig(
    phases=[OrganicFlowPhase()],
    seed=42,
    scenario_id="workbook_run",
)
environment = Environment(resolved, env_config)
simulation_log = environment.run()
log_tables = simulation_log.to_dataframes()
inventory_log = log_tables["simulation_inventory_log"]
flow_log = log_tables["simulation_flow_log"]
unmet_demand_log = log_tables["simulation_unmet_demand_log"]

In [20]:
temporal['planning_horizon']

,planning_horizon_id,name,start_date,end_date
0,h1,mock_horizon,2025-01-01,2025-01-04


In [14]:
resolved.facilities 

,facility_id,facility_type,name,lat,lon
0,4000.01,station,4000.01,40.777396,-74.007189
1,4013.02,station,4013.02,40.743888,-73.974961
2,4026.03,station,4026.03,40.785860,-73.982920
3,4039.04,station,4039.04,40.769737,-73.927324
4,4052.05,station,4052.05,40.709418,-73.955613
5,4065.06,station,4065.06,40.797562,-73.937724
6,4078.07,station,4078.07,40.776114,-73.975659
7,4091.08,station,4091.08,40.778606,-73.997276


In [17]:
resolved.edge_rules 

,source_type,target_type,commodity_category,modal_type,enabled
0,station,station,classic_bike,road,True
1,station,station,electric_bike,road,True


In [ ]:
# %% 7. Interactive dashboard
# Two buttons + one dropdown:
#   Generate        -> runs the minimal loader + build_model, draws stations on
#                      the map with their trip-origin counts (the only
#                      pre-simulation signal available in the minimal case).
#   Run Simulation  -> runs the Environment on the last generated model and
#                      replaces the map with simulated inventory per period.
#                      The period slider becomes active.
#   Bike type       -> filters both trip origins and simulated inventory by
#                      commodity_category (all / electric_bike / classic_bike).
#                      Re-renders the current map on change.

def _full_pipeline(n_stations: int, n_trips: int, n_days: int, seed: int):
    """Dashboard helper: runs the minimal loader -> build_model path."""
    config = {
        "n_stations": int(n_stations),
        "n_trips": int(n_trips),
        "n_days": int(n_days),
        "seed": int(seed),
    }
    source = DataLoaderMockMinimal(config)
    graph_loader = DataLoaderGraph(source)
    from gbp.build.pipeline import build_model                 # use the public entry point
    return build_model(graph_loader.load())

def _run_simulation(resolved_model) -> pd.DataFrame:
    env_cfg = EnvironmentConfig(
        phases=[OrganicFlowPhase()],
        seed=42,
        scenario_id="gradio_run",
    )
    return (
        Environment(resolved_model, env_cfg)
        .run()
        .to_dataframes()["simulation_inventory_log"]
    )

def _filter_by_bike_type(df: pd.DataFrame, bike_type: str) -> pd.DataFrame:
    """Keep only rows for the requested commodity_category (or all)."""
    if df is None or df.empty or bike_type == "all" or "commodity_category" not in df.columns:
        return df
    return df[df["commodity_category"] == bike_type]

def _trip_origin_counts(observed_flow: pd.DataFrame | None, bike_type: str = "all") -> pd.DataFrame:
    """Count organic outflow per source_id - baseline 'activity' signal."""
    if observed_flow is None or observed_flow.empty:
        return pd.DataFrame(columns=["facility_id", "quantity"])
    organic = observed_flow[observed_flow["target_id"].isna()] if "target_id" in observed_flow.columns else observed_flow
    if organic.empty:
        organic = observed_flow
    filtered = _filter_by_bike_type(organic, bike_type)
    if filtered.empty:
        return pd.DataFrame(columns=["facility_id", "quantity"])
    grouped = (
        filtered.groupby("source_id", as_index=False)["quantity"].sum()
        .rename(columns={"source_id": "facility_id"})
    )
    return grouped

def _inventory_at_period(log_df: pd.DataFrame, period_id: str, bike_type: str = "all") -> pd.DataFrame:
    if log_df is None or log_df.empty:
        return pd.DataFrame(columns=["facility_id", "quantity"])
    snapshot = log_df[log_df["period_id"] == period_id]
    snapshot = _filter_by_bike_type(snapshot, bike_type)
    if snapshot.empty:
        return pd.DataFrame(columns=["facility_id", "quantity"])
    return snapshot.groupby("facility_id", as_index=False)["quantity"].sum()

def _bike_type_suffix(bike_type: str) -> str:
    return "" if bike_type == "all" else f" [{bike_type}]"

def _make_map(
    facilities: pd.DataFrame,
    quantity_df: pd.DataFrame,
    title: str,
    quantity_label: str,
) -> go.Figure:
    merged = facilities.drop_duplicates("facility_id").merge(
        quantity_df.rename(columns={"quantity": "value"}),
        on="facility_id", how="left",
    )
    merged["value"] = merged["value"].fillna(0)

    color_map = {"station": "#1f77b4", "depot": "#d62728"}
    merged["marker_size"] = np.clip(merged["value"] * 0.5 + 6, 6, 40)

    figure = go.Figure()
    for facility_type, group in merged.groupby("facility_type"):
        customdata = np.stack([group["name"], group["value"]], axis=-1)
        figure.add_trace(go.Scattermap(
            lat=group["lat"], lon=group["lon"],
            mode="markers+text",
            marker=dict(size=group["marker_size"], color=color_map.get(facility_type, "#7f7f7f")),
            text=group["value"].round(0).astype(int).astype(str),
            textposition="top center",
            customdata=customdata,
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                f"{quantity_label}: " + "%{customdata[1]:.0f}<extra></extra>"
            ),
            name=str(facility_type),
        ))

    figure.update_layout(
        title=title,
        map=dict(
            style="open-street-map",
            center=dict(lat=merged["lat"].mean(), lon=merged["lon"].mean()),
            zoom=11,
        ),
        height=600, margin=dict(l=0, r=0, t=40, b=0), showlegend=True,
    )
    return figure

_dashboard_state = {
    "resolved": None, "facilities": None, "observed_flow": None,
    "simulation_log": None, "period_map": {},
}

def _generate(n_stations, n_trips, n_days, seed, bike_type):
    resolved_model = _full_pipeline(n_stations, n_trips, n_days, seed)
    facilities_df = resolved_model.facilities[
        ["facility_id", "facility_type", "name", "lat", "lon"]
    ].copy()
    observed_flow_df = resolved_model.observed_flow

    period_df = resolved_model.periods[["period_id", "start_date"]].sort_values("period_id")
    period_map = dict(zip(period_df["period_id"], period_df["start_date"].astype(str)))

    _dashboard_state.update(
        resolved=resolved_model, facilities=facilities_df,
        observed_flow=observed_flow_df, simulation_log=None, period_map=period_map,
    )

    trip_origins = _trip_origin_counts(observed_flow_df, bike_type)
    figure = _make_map(
        facilities_df, trip_origins,
        f"Stations & trip origins ({len(facilities_df)} facilities){_bike_type_suffix(bike_type)}",
        "trips from",
    )
    status = (
        f"Generated: {len(facilities_df)} facilities, {len(period_map)} periods, "
        f"{int(trip_origins['quantity'].sum()) if not trip_origins.empty else 0} organic trips"
        f"{_bike_type_suffix(bike_type)}. Click 'Run Simulation' to compute inventory per period."
    )
    slider_update = gr.update(minimum=0, maximum=max(len(period_map) - 1, 0), value=0, visible=False)
    return figure, status, slider_update

def _run_dashboard_simulation(bike_type):
    if _dashboard_state["resolved"] is None:
        return go.Figure(), "No data - click 'Generate' first", gr.update(visible=False)
    simulation_log_df = _run_simulation(_dashboard_state["resolved"])
    _dashboard_state["simulation_log"] = simulation_log_df

    period_map = _dashboard_state["period_map"]
    first_pid = next(iter(period_map))
    label = f"{first_pid} ({period_map[first_pid]})"
    figure = _make_map(
        _dashboard_state["facilities"],
        _inventory_at_period(simulation_log_df, first_pid, bike_type),
        f"Simulated inventory - {label}{_bike_type_suffix(bike_type)}",
        "inventory",
    )
    status = (
        f"Simulation complete: {len(simulation_log_df)} log rows. "
        "Use the period slider to browse."
    )
    slider_update = gr.update(
        minimum=0, maximum=len(period_map) - 1, value=0, visible=True,
    )
    return figure, status, slider_update

def _on_period_change(period_idx, bike_type):
    period_ids = list(_dashboard_state["period_map"].keys())
    if (
        not period_ids
        or _dashboard_state["facilities"] is None
        or _dashboard_state["simulation_log"] is None
    ):
        return go.Figure(), "No simulation data"
    period_idx = min(int(period_idx), len(period_ids) - 1)
    pid = period_ids[period_idx]
    label = f"{pid} ({_dashboard_state['period_map'][pid]})"

    figure = _make_map(
        _dashboard_state["facilities"],
        _inventory_at_period(_dashboard_state["simulation_log"], pid, bike_type),
        f"Simulated inventory - {label}{_bike_type_suffix(bike_type)}",
        "inventory",
    )
    return figure, f"Period {period_idx}: {label}{_bike_type_suffix(bike_type)}"

def _on_bike_type_change(bike_type, period_idx):
    """Re-render the currently displayed map (simulation if present, else trip origins)."""
    facilities_df = _dashboard_state["facilities"]
    if facilities_df is None:
        return go.Figure(), "No data - click 'Generate' first"

    simulation_log_df = _dashboard_state["simulation_log"]
    if simulation_log_df is not None:
        period_ids = list(_dashboard_state["period_map"].keys())
        period_idx = min(int(period_idx), len(period_ids) - 1) if period_ids else 0
        pid = period_ids[period_idx] if period_ids else None
        label = f"{pid} ({_dashboard_state['period_map'][pid]})" if pid else ""
        figure = _make_map(
            facilities_df,
            _inventory_at_period(simulation_log_df, pid, bike_type),
            f"Simulated inventory - {label}{_bike_type_suffix(bike_type)}",
            "inventory",
        )
        return figure, f"Bike type: {bike_type} - period {label}"

    trip_origins = _trip_origin_counts(_dashboard_state["observed_flow"], bike_type)
    figure = _make_map(
        facilities_df, trip_origins,
        f"Stations & trip origins ({len(facilities_df)} facilities){_bike_type_suffix(bike_type)}",
        "trips from",
    )
    total = int(trip_origins["quantity"].sum()) if not trip_origins.empty else 0
    return figure, f"Bike type: {bike_type} - {total} organic trips"

with gr.Blocks(title="Bike-Share Workbook - Minimal") as dashboard:
    gr.Markdown("## Bike-Share Network - Minimal Input, Unrolled Pipeline")
    with gr.Row():
        with gr.Column(scale=1):
            slider_stations = gr.Slider(3, 30, value=8, step=1, label="Stations")
            slider_trips = gr.Slider(20, 1000, value=200, step=20, label="Trips")
            slider_days = gr.Slider(1, 14, value=3, step=1, label="Days of history")
            slider_seed = gr.Slider(0, 999, value=42, step=1, label="Seed")
            dropdown_bike_type = gr.Dropdown(
                choices=["all", "electric_bike", "classic_bike"],
                value="all",
                label="Bike type (commodity_category)",
            )
            button_generate = gr.Button("Generate Mock Data", variant="primary")
            button_simulate = gr.Button("Run Simulation", variant="secondary")
            slider_period = gr.Slider(0, 1, value=0, step=1, label="Period", visible=False)
            status_text = gr.Textbox(label="Status", interactive=False)
        with gr.Column(scale=3):
            map_plot = gr.Plot(label="Network Map")

    button_generate.click(
        fn=_generate,
        inputs=[slider_stations, slider_trips, slider_days, slider_seed, dropdown_bike_type],
        outputs=[map_plot, status_text, slider_period],
    )
    button_simulate.click(
        fn=_run_dashboard_simulation,
        inputs=[dropdown_bike_type], outputs=[map_plot, status_text, slider_period],
    )
    slider_period.change(
        fn=_on_period_change,
        inputs=[slider_period, dropdown_bike_type],
        outputs=[map_plot, status_text],
    )
    dropdown_bike_type.change(
        fn=_on_bike_type_change,
        inputs=[dropdown_bike_type, slider_period],
        outputs=[map_plot, status_text],
    )

dashboard.launch()